# EEG_14 — DHSLP: Pretrain S-Indep + Fine-Tune S-Spec

**Ipotesi**: un modello pretrainato su tutti i 50 soggetti di training ha già appreso
rappresentazioni EEG generaliste. Fine-tuning per soggetto con LR basso dovrebbe
adattare queste rappresentazioni allo stile neurale specifico senza overfitting.

**Confronto atteso vs EEG_13b** (DHSLP S-Spec da zero):
- Il pretrain fornisce inizializzazione migliore rispetto a random init
- Fine-tuning richiede meno epoche / dati per convergere
- Atteso miglioramento su soggetti con pochi trial (5+ sessioni)

**Pipeline**:
```
§0  Config + imports
§1  Dataset (S-Indep per pretrain + LOSO per fine-tune)
§2  Modello DHSLP (identico a EEG_13/13b)
§3  §3a Pretrain S-Indep  →  salva models/eeg14/pretrain_E16_d64.pt
§4  Fine-tune per soggetto LOSO  →  LR=1e-4, carica pretrain ckpt
§5  Loop su tutti i soggetti
§6  Ricarica da checkpoint + ranking
§7  Bar chart ranking
§8  Confronto EEG_14 vs EEG_13b — delta plot
§9  Analisi soggetti 'sleeper' e interpretazione
```

**Config pretrain**: N_EDGES=16, D_MODEL=64 (best da ablation EEG_13, test bAcc=25.68%)  
**Split S-Indep**: SUBJ_TRAIN=0–49, SUBJ_VAL=50–59, SUBJ_TEST=60–73  
**Split LOSO**: test=ultima sessione, val=penultima, train=resto (come EEG_13b)

In [ ]:
import json, logging, re, traceback
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score, recall_score
from tqdm.auto import tqdm
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg14')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg14'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ---- CONFIG CONDIVISA ----
N_CHANNELS   = 61
N_SAMPLES    = 384
N_CLASSES    = 4
CLUSTER_SCHEME = 'concr4'

# Architettura DHSLP — best config da EEG_13 ablation (E16_d64 = 25.68% test bAcc)
K_WINDOWS  = 8
N_EDGES    = 16
D_MODEL    = 64
HIDDEN     = 128
N_LAYERS   = 2
T_WIN      = N_SAMPLES // K_WINDOWS   # 48

# --- CONFIG PRETRAIN (S-Indep, identica a EEG_13) ---
PRETRAIN_DROPOUT       = 0.3
PRETRAIN_LR            = 1e-3
PRETRAIN_WEIGHT_DECAY  = 1e-4
PRETRAIN_BATCH_SIZE    = 64
PRETRAIN_MAX_EPOCHS    = 60
PRETRAIN_PATIENCE      = 12
PRETRAIN_LABEL_SMOOTH  = 0.1

SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Percorso checkpoint pretrain (salvato da §3a, caricato da §4)
PRETRAIN_CKPT = CKPT_DIR / 'pretrain_E16_d64.pt'

# --- CONFIG FINE-TUNE (S-Spec LOSO) ---
FT_DROPOUT       = 0.5    # alta regolarizzazione: ~330 trial/soggetto
FT_LR            = 1e-4   # LR basso: non sovrascrivo pesi pretrain
FT_WEIGHT_DECAY  = 1e-2   # forte L2: contrasta memorizzazione
FT_BATCH_SIZE    = 32
FT_MAX_EPOCHS    = 40     # meno epoche: la rete parte già da buona init
FT_PATIENCE      = 8
FT_LABEL_SMOOTH  = 0.1
USE_INSTANCE_NORM = True

DATA_METRIC = 'abs_pcc'   # path arbitrario — usiamo solo x e y (H ignorata da DHSLP)

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

# Indicizza tutti i trial per soggetto e sessione
HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
subj_sess = defaultdict(lambda: defaultdict(list))
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess[int(m.group(1))][int(m.group(2))].append(p)

ALL_SUBJ = sorted(subj_sess.keys())
log.info(f'HG_ROOT: {HG_ROOT}')
log.info(f'Soggetti trovati: {len(ALL_SUBJ)}  T_WIN={T_WIN}  K={K_WINDOWS}')
log.info(f'N_EDGES={N_EDGES}  D_MODEL={D_MODEL}  (best da EEG_13 ablation)')
log.info(f'PRETRAIN_CKPT: {PRETRAIN_CKPT}')
log.info(f'FT_LR={FT_LR}  FT_WEIGHT_DECAY={FT_WEIGHT_DECAY}  FT_DROPOUT={FT_DROPOUT}')

## §1 — Dataset

In [ ]:
# ---- Dataset S-Indep (per pretrain) ----

class EEGRawDataset(Dataset):
    """
    Carica x (61,384) e y dai file .pt preprocessati.
    H_pruned ignorata — DHSLP apprende H internamente.
    Usato per pretrain S-Indep (split SUBJ_TRAIN/VAL/TEST).
    """
    def __init__(self, subj_ids, metric=DATA_METRIC, use_instance_norm=True):
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.paths, self.labels = [], []
        self.use_instance_norm = use_instance_norm
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            if int(m.group(1)) not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p)
            self.labels.append(c)
        log.info(f'  EEGRawDataset({len(subj_ids)} sogg.): {len(self.paths)} trial')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


def make_sindep_loaders():
    """Crea DataLoader S-Indep per pretrain."""
    tr = EEGRawDataset(SUBJ_TRAIN)
    va = EEGRawDataset(SUBJ_VAL)
    te = EEGRawDataset(SUBJ_TEST)
    labels   = np.array(tr.labels)
    counts   = np.bincount(labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / counts[labels], dtype=torch.float)
    sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, PRETRAIN_BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(va, PRETRAIN_BATCH_SIZE, shuffle=False, **kw),
            DataLoader(te, PRETRAIN_BATCH_SIZE, shuffle=False, **kw))


# ---- Dataset S-Spec LOSO (per fine-tune) ----

class EEGRawDatasetSS(Dataset):
    """
    Carica x (61,384) e y da lista di (path, label).
    Usato per fine-tuning S-Spec LOSO.
    """
    def __init__(self, paths_and_labels, use_instance_norm=True):
        self.items = paths_and_labels
        self.use_instance_norm = use_instance_norm

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        p, label = self.items[idx]
        d = torch.load(p, weights_only=False)
        x = d['x'].float()
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return x, torch.tensor(label, dtype=torch.long)


def _collect(subj_id, sess_list):
    items = []
    for s in sess_list:
        for p in subj_sess[subj_id][s]:
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is not None:
                items.append((p, c))
    return items


def make_loso_loaders(subj_id):
    """LOSO split: test=ultima sessione, val=penultima, train=resto."""
    sids = sorted(subj_sess[subj_id].keys())
    if len(sids) < 2:
        return None
    test_sess  = sids[-1]
    val_sess   = sids[-2]
    train_sess = [s for s in sids if s not in (test_sess, val_sess)]

    tr_items = _collect(subj_id, train_sess)
    va_items = _collect(subj_id, [val_sess])
    te_items = _collect(subj_id, [test_sess])
    if not tr_items or not te_items:
        return None

    tr_labels = np.array([it[1] for it in tr_items])
    counts    = np.bincount(tr_labels, minlength=N_CLASSES)
    sample_w  = torch.tensor(1.0 / counts[tr_labels], dtype=torch.float)
    sampler   = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    kw = dict(num_workers=0, pin_memory=False)
    return (
        DataLoader(EEGRawDatasetSS(tr_items, USE_INSTANCE_NORM), FT_BATCH_SIZE, sampler=sampler, **kw),
        DataLoader(EEGRawDatasetSS(va_items, USE_INSTANCE_NORM), FT_BATCH_SIZE, shuffle=False, **kw),
        DataLoader(EEGRawDatasetSS(te_items, USE_INSTANCE_NORM), FT_BATCH_SIZE, shuffle=False, **kw),
    )

## §2 — Modello DHSLP

In [ ]:
class HGNNConv(nn.Module):
    """HGNN spectral convolution layer (Feng et al. 2019) — batched."""
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)

    def forward(self, X, H):
        # X: (B,N,C)  H: (B,N,E) — soft incidence matrix dinamica
        d_v = H.sum(dim=2).clamp(min=1e-6)
        d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv  = (1.0 / d_v.sqrt()).unsqueeze(-1)
        De  = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out
        out = torch.bmm(H, out)
        out = Dv * out
        if self.bias is not None: out = out + self.bias
        return out


class DHSLP(nn.Module):
    """
    Dynamic Hypergraph Spectral Learning with Positional encoding.
    Ispirato a Li et al. 2025.

    Input:  x (B, 61, 384)  — nessuna H in input
    Output: logits (B, 4)

    Pipeline per finestra temporale k:
      feat_k = Linear(x_k) + pos_enc        → (B, 61, d)
      H_k    = softmax(feat_k @ E^T / √d)   → (B, 61, n_edges)  [DINAMICA]
      out_k  = HGNNConv(feat_k, H_k).mean   → (B, hidden)
    z = mean(out_1..K) → Linear → logits
    """
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS,
                 n_edges=N_EDGES, d_model=D_MODEL, hidden=HIDDEN,
                 n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=0.3):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model

        # Learnable iperedge embeddings — E ∈ R^{n_edges × d}
        self.E = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        # Encoding posizionale per elettrodo — pos_enc ∈ R^{61 × d}
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)

        self.node_proj = nn.Sequential(
            nn.Linear(T_win, d_model),
            nn.LayerNorm(d_model),
            nn.ELU(),
        )
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Linear(hidden, n_classes)

    def build_dynamic_H(self, feat):
        """feat: (B,N,d)  →  H: (B,N,n_edges) — soft assignment nodo→iperedge."""
        scores = torch.matmul(feat, self.E.T) / (self.d_model ** 0.5)
        return torch.softmax(scores, dim=2)

    def forward(self, x):
        B, N, T = x.shape
        outs = []
        for k in range(self.K):
            x_k  = x[:, :, k*self.T_win:(k+1)*self.T_win]   # (B,N,T_win)
            feat = self.node_proj(x_k) + self.pos_enc        # (B,N,d)
            H_k  = self.build_dynamic_H(feat)                # (B,N,n_edges)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out)
                out = self.drop(out)
            outs.append(out.mean(dim=1))   # (B, hidden)
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')

# Sanity check architettura
_m = DHSLP(dropout=PRETRAIN_DROPOUT)
_x = torch.randn(4, N_CHANNELS, N_SAMPLES)
assert _m(_x).shape == (4, N_CLASSES)
n_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
log.info(f'DHSLP OK — {n_p:,} parametri  K={K_WINDOWS} N_EDGES={N_EDGES} D_MODEL={D_MODEL}')
del _m, _x

## §3 — Pretrain S-Indep

**Identico a EEG_13** (SUBJ_TRAIN=0–49). Salva checkpoint su disco per il fine-tuning.

Se `PRETRAIN_CKPT` esiste già, questo step può essere saltato (`SKIP_PRETRAIN = True`).

In [ ]:
SKIP_PRETRAIN = PRETRAIN_CKPT.exists()   # True se checkpoint già presente su disco

if SKIP_PRETRAIN:
    log.info(f'Pretrain checkpoint già presente: {PRETRAIN_CKPT} — skip §3.')
    log.info('Per ri-addestrare da zero: cancella il file o imposta SKIP_PRETRAIN=False.')
else:
    log.info('=== PRETRAIN S-Indep (EEG_14 §3) ===')
    log.info(f'SUBJ_TRAIN={len(SUBJ_TRAIN)}  VAL={len(SUBJ_VAL)}  TEST={len(SUBJ_TEST)}')

    pt_tr_loader, pt_va_loader, pt_te_loader = make_sindep_loaders()

    pretrain_model = DHSLP(dropout=PRETRAIN_DROPOUT).to(device)
    pt_criterion   = nn.CrossEntropyLoss(label_smoothing=PRETRAIN_LABEL_SMOOTH)
    pt_optimizer   = torch.optim.Adam(pretrain_model.parameters(),
                                      lr=PRETRAIN_LR, weight_decay=PRETRAIN_WEIGHT_DECAY)
    pt_sched       = torch.optim.lr_scheduler.CosineAnnealingLR(pt_optimizer, T_max=PRETRAIN_MAX_EPOCHS)

    pt_run = wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name=f'eeg14_pretrain_DHSLP_{CLUSTER_SCHEME}',
        config=dict(
            notebook='EEG_14', phase='pretrain', model='DHSLP_SIndep',
            n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
            k_windows=K_WINDOWS, t_win=T_WIN,
            n_edges=N_EDGES, d_model=D_MODEL,
            hidden=HIDDEN, n_layers=N_LAYERS, dropout=PRETRAIN_DROPOUT,
            lr=PRETRAIN_LR, weight_decay=PRETRAIN_WEIGHT_DECAY,
            batch_size=PRETRAIN_BATCH_SIZE, max_epochs=PRETRAIN_MAX_EPOCHS,
            n_train_subj=len(SUBJ_TRAIN), n_val_subj=len(SUBJ_VAL),
            use_instance_norm=USE_INSTANCE_NORM,
            label_smoothing=PRETRAIN_LABEL_SMOOTH,
        ),
        reinit='finish_previous',
        settings=wandb.Settings(start_method='thread')
    )

    pt_best_val, pt_best_state, pt_patience = 0.0, None, 0

    def _run_epoch_pt(model, loader, optimizer=None):
        train = optimizer is not None
        model.train() if train else model.eval()
        total_loss, all_lbl, all_pred = 0.0, [], []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss   = pt_criterion(logits, y)
                if train:
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                total_loss += loss.item() * len(y)
                all_lbl.extend(y.cpu().numpy())
                all_pred.extend(logits.argmax(1).cpu().numpy())
        bacc = balanced_accuracy_score(all_lbl, all_pred)
        return total_loss / len(loader.dataset), bacc

    for epoch in range(1, PRETRAIN_MAX_EPOCHS + 1):
        tr_loss, tr_b = _run_epoch_pt(pretrain_model, pt_tr_loader, pt_optimizer)
        va_loss, va_b = _run_epoch_pt(pretrain_model, pt_va_loader)
        pt_sched.step()
        pt_run.log({'train/loss': tr_loss, 'train/bacc': tr_b,
                    'val/loss': va_loss,   'val/bacc': va_b, 'epoch': epoch})
        if va_b > pt_best_val:
            pt_best_val = va_b
            pt_best_state = {k: v.cpu().clone() for k, v in pretrain_model.state_dict().items()}
            pt_patience = 0
        else:
            pt_patience += 1
        if pt_patience >= PRETRAIN_PATIENCE:
            log.info(f'  Pretrain early stop @ epoch {epoch}'); break
        if epoch % 10 == 0:
            log.info(f'  Epoch {epoch:3d}: train_bAcc={tr_b:.4f}  val_bAcc={va_b:.4f}  best_val={pt_best_val:.4f}')

    pretrain_model.load_state_dict(pt_best_state)
    _, pt_te_b = _run_epoch_pt(pretrain_model, pt_te_loader)
    pt_run.summary['val_bacc']  = pt_best_val
    pt_run.summary['test_bacc'] = pt_te_b
    pt_run.finish()

    # Salva checkpoint su disco — verrà caricato da tutti i soggetti in §4
    torch.save({'state_dict': pt_best_state,
                'val_bacc': pt_best_val, 'test_bacc': pt_te_b,
                'config': {'n_edges': N_EDGES, 'd_model': D_MODEL,
                           'k_windows': K_WINDOWS, 'hidden': HIDDEN, 'n_layers': N_LAYERS}},
               PRETRAIN_CKPT)
    log.info(f'Pretrain salvato: {PRETRAIN_CKPT}')
    log.info(f'Pretrain best_val={pt_best_val:.4f}  test_bAcc={pt_te_b:.4f}')

## §4 — Fine-Tune per Soggetto

**Strategia**: carica pesi pretrain → re-inizializza `dropout` a `FT_DROPOUT=0.5` → ottimizza con `LR=1e-4`.

Fine-tuning full-network (tutti i pesi, inclusi E e pos_enc): il basso LR protegge le
rappresentazioni generaliste apprese durante il pretrain.

In [ ]:
ft_criterion = nn.CrossEntropyLoss(label_smoothing=FT_LABEL_SMOOTH)


def run_epoch_ft(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_lbl, all_pred = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = ft_criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            all_lbl.extend(y.cpu().numpy())
            all_pred.extend(logits.argmax(1).cpu().numpy())
    bacc = balanced_accuracy_score(all_lbl, all_pred)
    return total_loss / len(loader.dataset), bacc, np.array(all_lbl), np.array(all_pred)


def load_pretrain_model():
    """
    Carica il modello DHSLP con i pesi pretrain e sostituisce dropout con FT_DROPOUT.
    Restituisce modello pronto per fine-tuning.
    """
    if not PRETRAIN_CKPT.exists():
        raise FileNotFoundError(f'Checkpoint pretrain non trovato: {PRETRAIN_CKPT}\nEsegui prima §3.')
    ckpt = torch.load(PRETRAIN_CKPT, weights_only=False)
    # Crea modello con dropout da fine-tune (diverso dal pretrain)
    model = DHSLP(dropout=FT_DROPOUT)
    model.load_state_dict(ckpt['state_dict'])
    return model.to(device)


def finetune_subject(subj_id, tr_l, va_l, te_l):
    run_name = f'eeg14_DHSLP_PT_P{subj_id:03d}_{CLUSTER_SCHEME}'
    cfg = dict(
        notebook='EEG_14', phase='finetune', model='DHSLP_PT',
        subject=f'P{subj_id:03d}',
        n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
        k_windows=K_WINDOWS, t_win=T_WIN,
        n_edges=N_EDGES, d_model=D_MODEL,
        hidden=HIDDEN, n_layers=N_LAYERS, dropout=FT_DROPOUT,
        lr=FT_LR, weight_decay=FT_WEIGHT_DECAY,
        batch_size=FT_BATCH_SIZE, max_epochs=FT_MAX_EPOCHS,
        use_instance_norm=USE_INSTANCE_NORM,
        label_smoothing=FT_LABEL_SMOOTH,
        weighted_sampler=True,
        pretrain_ckpt=str(PRETRAIN_CKPT),
        n_train=len(tr_l.dataset),
    )
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=run_name, config=cfg, reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))

    model = load_pretrain_model()
    opt   = torch.optim.Adam(model.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_MAX_EPOCHS)
    best_val, best_state, patience_cnt = 0.0, None, 0

    for epoch in range(1, FT_MAX_EPOCHS + 1):
        tr_loss, tr_b, _, _ = run_epoch_ft(model, tr_l, opt)
        va_loss, va_b, _, _ = run_epoch_ft(model, va_l)
        sched.step()
        run.log({'train/loss': tr_loss, 'train/bacc': tr_b,
                 'val/loss': va_loss,   'val/bacc': va_b, 'epoch': epoch})
        if va_b > best_val:
            best_val = va_b
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= FT_PATIENCE:
            log.info(f'  P{subj_id:03d} early stop @ epoch {epoch}'); break

    model.load_state_dict(best_state)
    _, te_b, te_lbl, te_pred = run_epoch_ft(model, te_l)

    # Checkpoint per soggetto
    ckpt_path = CKPT_DIR / f'P{subj_id:03d}.pt'
    torch.save({'state_dict': best_state, 'val_bacc': best_val, 'test_bacc': te_b,
                'labels': te_lbl, 'preds': te_pred}, ckpt_path)

    run.summary['val_bacc']  = best_val
    run.summary['test_bacc'] = te_b
    run.log({'confusion_matrix': wandb.plot.confusion_matrix(
        preds=te_pred.tolist(), y_true=te_lbl.tolist(),
        class_names=['CONCR', 'AZIONE', 'STATO', 'ASTRATTO'])})
    run.finish()
    log.info(f'  P{subj_id:03d}: val={best_val:.4f}  test={te_b:.4f}')
    return {'val_bacc': best_val, 'test_bacc': te_b, 'labels': te_lbl, 'preds': te_pred}

## §5 — Loop su Tutti i Soggetti

In [ ]:
SUBJECT_RESULTS = {}

if not PRETRAIN_CKPT.exists():
    log.error(f'Checkpoint pretrain non trovato: {PRETRAIN_CKPT}')
    log.error('Esegui prima §3 (pretrain S-Indep) per generare il checkpoint.')
else:
    log.info(f'=== FINE-TUNE S-Spec LOSO — {len(ALL_SUBJ)} soggetti ===')
    log.info(f'Pretrain ckpt: {PRETRAIN_CKPT}')

    for sid in tqdm(ALL_SUBJ, desc='DHSLP Fine-Tune'):
        loaders = make_loso_loaders(sid)
        if loaders is None:
            log.warning(f'P{sid:03d}: skip (sessioni insufficienti)')
            continue
        tr_l, va_l, te_l = loaders
        log.info(f'P{sid:03d}: train={len(tr_l.dataset)} val={len(va_l.dataset)} test={len(te_l.dataset)}')
        try:
            SUBJECT_RESULTS[sid] = finetune_subject(sid, tr_l, va_l, te_l)
        except Exception as e:
            log.error(f'P{sid:03d}: {e}\n{traceback.format_exc()}')

    log.info(f'\n=== DONE: {len(SUBJECT_RESULTS)}/{len(ALL_SUBJ)} soggetti ===')

## §6 — Ricarica da Checkpoint + Ranking

In [ ]:
if not SUBJECT_RESULTS:
    log.info('Ricarico da checkpoint...')
    for ckpt in sorted(CKPT_DIR.glob('P*.pt')):
        sid = int(ckpt.stem[1:])
        d = torch.load(ckpt, weights_only=False)
        SUBJECT_RESULTS[sid] = {k: d[k] for k in ('val_bacc', 'test_bacc', 'labels', 'preds')}
    log.info(f'Ricaricati {len(SUBJECT_RESULTS)} soggetti')

if not SUBJECT_RESULTS:
    print('[INFO] Nessun risultato — esegui prima §3 + §5.')
else:
    def top2_bacc(labels, preds, n_classes=N_CLASSES):
        recalls = recall_score(labels, preds, average=None, zero_division=0,
                               labels=list(range(n_classes)))
        top2_cls = np.argsort(recalls)[-2:]
        mask = np.isin(labels, top2_cls)
        if mask.sum() == 0: return np.nan, top2_cls.tolist()
        return balanced_accuracy_score(labels[mask], preds[mask]), top2_cls.tolist()

    rows = []
    for sid, res in SUBJECT_RESULTS.items():
        lbl = np.array(res['labels']); pred = np.array(res['preds'])
        t2, _ = top2_bacc(lbl, pred)
        rows.append({
            'Subject':   f'P{sid:03d}',
            'Test bAcc': round(res['test_bacc'], 4),
            'Top2 bAcc': round(t2, 4) if not np.isnan(t2) else np.nan,
        })

    df_res = pd.DataFrame(rows).sort_values('Test bAcc', ascending=False).reset_index(drop=True)
    df_res.to_csv(FIG_DIR / 'eeg14_subject_ranking.csv', index=False)

    chance = 1 / N_CLASSES
    print('Top-10 by Test bAcc (DHSLP Pretrain+Fine-Tune):')
    print(df_res.head(10).to_string(index=False))
    print(f'\nMediana: {df_res["Test bAcc"].median():.4f}  '
          f'Mean: {df_res["Test bAcc"].mean():.4f}  '
          f'Max: {df_res["Test bAcc"].max():.4f}')
    print(f'% sopra chance: {(df_res["Test bAcc"] > chance).mean()*100:.1f}%')

## §7 — Bar Chart Ranking

In [ ]:
if 'df_res' not in dir():
    print('[INFO] Esegui prima §6.')
else:
    chance = 1 / N_CLASSES

    fig, ax = plt.subplots(figsize=(18, 5))
    fig.suptitle('EEG_14 — DHSLP Pretrain + Fine-Tune (Subject-Specific LOSO)', fontsize=13, fontweight='bold')

    colors = ['#2C7BB6' if v > chance else '#D7191C' for v in df_res['Test bAcc']]
    bars = ax.bar(range(len(df_res)), df_res['Test bAcc'], color=colors, width=0.8)
    ax.axhline(chance, color='black', linestyle='--', linewidth=1.5, label=f'Chance ({chance:.2%})')
    ax.set_xticks(range(len(df_res)))
    ax.set_xticklabels(df_res['Subject'], rotation=90, fontsize=7)
    ax.set_xlabel('Soggetto (ordinato per test bAcc)')
    ax.set_ylabel('Balanced Accuracy')
    ax.set_title('Test bAcc per soggetto  (blu = sopra chance, rosso = sotto)')
    ax.set_ylim(0, max(df_res['Test bAcc'].max() + 0.05, 0.55))
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    out = FIG_DIR / 'eeg14_subject_ranking.png'
    plt.savefig(out, dpi=150)
    plt.show()
    log.info(f'Salvato: {out}')

## §8 — Confronto EEG_14 vs EEG_13b

**Delta plot**: EEG_14 (pretrain+FT) − EEG_13b (da zero) per ogni soggetto.
- Barra **verde** (Δ > 0): il pretrain ha aiutato
- Barra **rossa** (Δ < 0): fine-tuning ha peggiorato rispetto a training da zero

Tabella laterale: top-15 soggetti con maggior miglioramento.

In [ ]:
# Carica risultati EEG_13b da checkpoint
CKPT_DIR_13B = project_root / 'models' / 'eeg13b'

results_13b = {}
if CKPT_DIR_13B.exists():
    for ckpt in sorted(CKPT_DIR_13B.glob('P*.pt')):
        sid = int(ckpt.stem[1:])
        d = torch.load(ckpt, weights_only=False)
        results_13b[sid] = d.get('test_bacc', np.nan)
    log.info(f'EEG_13b: {len(results_13b)} soggetti ricaricati')
else:
    log.warning(f'Directory EEG_13b non trovata: {CKPT_DIR_13B}')
    log.warning('Confronto non disponibile. Esegui EEG_13b prima di questo step.')

if 'df_res' not in dir() or not results_13b:
    print('[INFO] Esegui §6 (EEG_14) e assicurati che i checkpoint EEG_13b esistano.')
else:
    # Costruisci dataframe di confronto su soggetti comuni
    common_sids = sorted(set(SUBJECT_RESULTS.keys()) & set(results_13b.keys()))
    rows_cmp = []
    for sid in common_sids:
        acc14  = SUBJECT_RESULTS[sid]['test_bacc']
        acc13b = results_13b[sid]
        rows_cmp.append({
            'Subject': f'P{sid:03d}',
            'EEG_13b': round(acc13b, 4),
            'EEG_14':  round(acc14, 4),
            'Delta':   round(acc14 - acc13b, 4),
        })

    df_cmp = pd.DataFrame(rows_cmp).sort_values('Delta', ascending=False).reset_index(drop=True)
    df_cmp.to_csv(FIG_DIR / 'eeg14_vs_13b_comparison.csv', index=False)

    n_improved = (df_cmp['Delta'] > 0).sum()
    n_degraded = (df_cmp['Delta'] < 0).sum()
    mean_delta = df_cmp['Delta'].mean()
    median_delta = df_cmp['Delta'].median()

    print(f'Soggetti comuni: {len(df_cmp)}')
    print(f'Migliorati (Δ>0): {n_improved}  Peggiorati (Δ<0): {n_degraded}')
    print(f'Media Δ: {mean_delta:+.4f}   Mediana Δ: {median_delta:+.4f}')
    print(f'Max Δ: {df_cmp["Delta"].max():+.4f}   Min Δ: {df_cmp["Delta"].min():+.4f}')
    print('\nTop-10 soggetti con maggior miglioramento:')
    print(df_cmp.head(10).to_string(index=False))

    # ---- Delta plot ----
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle('EEG_14 vs EEG_13b — Δ test bAcc (Pretrain+FT − Training da zero)', fontsize=13, fontweight='bold')

    # Panel sinistro: delta bar chart
    df_sorted = df_cmp.sort_values('Delta', ascending=False)
    bar_colors = ['#2ca02c' if d > 0 else '#d62728' for d in df_sorted['Delta']]
    ax1.bar(range(len(df_sorted)), df_sorted['Delta'], color=bar_colors, width=0.8)
    ax1.axhline(0, color='black', linewidth=1.0)
    ax1.set_xticks(range(len(df_sorted)))
    ax1.set_xticklabels(df_sorted['Subject'], rotation=90, fontsize=7)
    ax1.set_xlabel('Soggetto')
    ax1.set_ylabel('Δ bAcc')
    ax1.set_title(f'Delta per soggetto  (verde=migliorato, rosso=peggiorato)\n'
                  f'Media Δ={mean_delta:+.4f}  Migliorati={n_improved}/{len(df_cmp)}')
    ax1.grid(axis='y', alpha=0.3)

    # Panel destro: scatter EEG_13b vs EEG_14
    chance = 1 / N_CLASSES
    ax2.scatter(df_cmp['EEG_13b'], df_cmp['EEG_14'], alpha=0.7, s=40, color='#1f77b4')
    for _, row in df_cmp.iterrows():
        if abs(row['Delta']) > 0.03:
            ax2.annotate(row['Subject'], (row['EEG_13b'], row['EEG_14']),
                         textcoords='offset points', xytext=(3, 3), fontsize=7)
    lims = [min(df_cmp[['EEG_13b','EEG_14']].values.min(), chance) - 0.01,
            max(df_cmp[['EEG_13b','EEG_14']].values.max(), chance) + 0.01]
    ax2.plot(lims, lims, 'k--', linewidth=1, label='y=x (nessuna variazione)')
    ax2.axvline(chance, color='gray', linestyle=':', linewidth=1)
    ax2.axhline(chance, color='gray', linestyle=':', linewidth=1)
    ax2.set_xlim(lims); ax2.set_ylim(lims)
    ax2.set_xlabel('EEG_13b bAcc (da zero)')
    ax2.set_ylabel('EEG_14 bAcc (pretrain+FT)')
    ax2.set_title('Scatter: punti sopra la diagonale = pretrain aiuta')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    out = FIG_DIR / 'eeg14_vs_13b_comparison.png'
    plt.savefig(out, dpi=150)
    plt.show()
    log.info(f'Salvato: {out}')

## §9 — Analisi Soggetti 'Sleeper' e Interpretazione

I soggetti per cui il pretrain aiuta di più sono potenzialmente "universally decodable" —
il loro segnale EEG ha caratteristiche che il modello generalista ha già visto nel training set.

In [ ]:
if 'df_cmp' not in dir():
    print('[INFO] Esegui prima §8.')
else:
    chance = 1 / N_CLASSES

    # Soggetti che escono da chance solo con il pretrain
    unlocked = df_cmp[(df_cmp['EEG_13b'] <= chance) & (df_cmp['EEG_14'] > chance)]
    # Soggetti che regrediscono sotto chance col fine-tuning
    regressed = df_cmp[(df_cmp['EEG_13b'] > chance) & (df_cmp['EEG_14'] <= chance)]
    # Soggetti sopra chance in entrambi i modelli
    both_above = df_cmp[(df_cmp['EEG_13b'] > chance) & (df_cmp['EEG_14'] > chance)]

    print('='*60)
    print('ANALISI COMPARATIVA EEG_14 vs EEG_13b')
    print('='*60)
    print(f'\n[A] Soggetti SBLOCCATI dal pretrain (era ≤ chance, ora > chance):')
    if len(unlocked) > 0:
        print(unlocked[['Subject','EEG_13b','EEG_14','Delta']].to_string(index=False))
    else:
        print('  Nessuno.')

    print(f'\n[B] Soggetti REGREDITI dal fine-tuning (era > chance, ora ≤ chance):')
    if len(regressed) > 0:
        print(regressed[['Subject','EEG_13b','EEG_14','Delta']].to_string(index=False))
    else:
        print('  Nessuno.')

    print(f'\n[C] Sopra chance in ENTRAMBI i modelli ({len(both_above)} soggetti):')
    print(both_above[['Subject','EEG_13b','EEG_14','Delta']].sort_values('EEG_14', ascending=False).to_string(index=False))

    print(f'\n[D] Statistiche aggregate:')
    print(f'  EEG_13b — Mean: {df_cmp["EEG_13b"].mean():.4f}  Median: {df_cmp["EEG_13b"].median():.4f}  % sopra chance: {(df_cmp["EEG_13b"] > chance).mean()*100:.1f}%')
    print(f'  EEG_14  — Mean: {df_cmp["EEG_14"].mean():.4f}  Median: {df_cmp["EEG_14"].median():.4f}  % sopra chance: {(df_cmp["EEG_14"] > chance).mean()*100:.1f}%')
    print(f'  Delta   — Mean: {df_cmp["Delta"].mean():+.4f}  Median: {df_cmp["Delta"].median():+.4f}')

    print('\n[INTERPRETAZIONE]')
    if df_cmp['Delta'].mean() > 0.005:
        print('  → Il pretrain migliora sistematicamente. Le rappresentazioni S-Indep')
        print('    sono un punto di partenza più informativo di random init.')
    elif df_cmp['Delta'].mean() < -0.005:
        print('  → Il fine-tuning peggiora rispetto a training da zero.')
        print('    Probabile causa: LR_FT ancora troppo alto, oppure')
        print('    le rappresentazioni S-Indep non sono allineate con il segnale individuale.')
    else:
        print('  → Delta ≈ 0: il pretrain non porta beneficio significativo.')
        print('    Interpretazione: con ~330 trial/sogg., i pesi S-Indep vengono')
        print('    rapidamente sovrascritti → same regime del training da zero.')
        print('    Possibile soluzione: freeze encoder, fine-tune solo clf head.')